# CD CosMx Biopsies: LIANA + NMF

Bandwidth selection → LIANA bivariate (cosine local + Moran's I; 100 permutations) → NMF spatial factorization (K=6).
Primary sample used in paper: CD_B4_slide1 (Fig 4).

**Set `SLIDE_ID` below to switch between samples.**
Available slides:
- `CD_B4_slide1` — primary inflamed biopsy (Fig 4, used in paper)
- `CD_B5_slide1` — validation inflamed biopsy (Ext Fig 9A-C)
- `CD_B4_slide3` — non-inflamed control (Ext Fig 9)
- `CD_B5_slide3` — non-inflamed control (Ext Fig 9)


In [ ]:
# ── Paths ──
DATA_DIR   = "/path/to/cosmx_data/cd_1k_biopsy"
OUTPUT_DIR = DATA_DIR + "/CD_B4/Processed"

In [ ]:
# ── Sample configuration ─────────────────────────────────────────────────────
# Set SLIDE_ID to select which biopsy slide to analyze.
# Options: "CD_B4_slide1" | "CD_B5_slide1" | "CD_B4_slide3" | "CD_B5_slide3"
SLIDE_ID = "CD_B4_slide1"  # primary inflamed biopsy used in paper (Fig 4)


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scanpy.external as sce
import liana as li
from scipy.spatial import distance_matrix
import pyreadr

In [ ]:
adata=sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/norm_anno_tregbin.h5ad')

In [ ]:
from liana.method import MistyData, genericMistyData, lrMistyData
from liana.method.sp import RandomForestModel, LinearModel, RobustLinearModel
import decoupler as dc
import plotnine as p9

In [ ]:
adata_slide = adata[adata.obs['slide'] == SLIDE_ID]

# deciding bandwidth for lr analysis

In [ ]:
# Copy spatial_fov to spatial, but flip the Y-axis
coords = adata_slide.obsm['spatial_fov'].copy()

# Flip Y coordinates
y_max = coords[:, 1].max()
y_min = coords[:, 1].min()
coords[:, 1] = y_max + y_min - coords[:, 1]  # Mirror around center

# Set as spatial
adata_slide.obsm['spatial'] = coords

In [ ]:
plot, _ = li.ut.query_bandwidth(coordinates=adata_slide.obsm['spatial'], start=0, end=500, interval_n=25)
plot

In [ ]:
li.ut.spatial_neighbors(adata_slide, bandwidth=200, cutoff=0.1, kernel='gaussian', set_diag=True, max_neighbours=100)

# running LR

In [ ]:
import requests
url = "https://zenodo.org/record/7074291/files/lr_network_human_21122021.rds"
r = requests.get(url)
with open("/path/to/nichenet/lr_network_human_21122021.rds", "wb") as f:
    f.write(r.content)

In [ ]:
# Now read it with pyreadr
lr = pyreadr.read_r("/path/to/nichenet/lr_network_human_21122021.rds")


In [ ]:
lr_df = next(iter(lr.values()))
lr_df

In [ ]:
lr_df=lr_df[['from','to']]

In [ ]:
lr_df.columns = ['ligand','receptor']

In [ ]:
lrdata_nn=sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/lr_liana_CD_B4_1_nn_db.h5ad')

In [ ]:
morans_sorted = lrdata_nn.var.sort_values("morans", ascending=False)

In [ ]:
morans_sorted['index_col'] = range(len(morans_sorted))

In [ ]:
from kneed import KneeLocator
vals = morans_sorted['morans'].values

x = np.arange(len(vals))

# --- only use points AFTER the 10th one for elbow search
start = 100
x_sub = x[start:]
y_sub = vals[start:]

# find elbow on the subset
kl = KneeLocator(
    x_sub, y_sub,
    curve='concave',        # choose based on shape
    direction='increasing'  # choose based on shape
)

print("Elbow (subset index):", kl.elbow)

# 3) plot
plt.figure(figsize=(6,4))
plt.plot(x, vals, marker='.', linewidth=1)

# highlight elbow
if kl.elbow is not None:
    plt.axvline(kl.elbow, color='red', linestyle='--', linewidth=2,
                label=f'Elbow = {kl.elbow}')
    plt.legend()

plt.xlabel("Sorted index")
plt.ylabel("Moran's I")
plt.title("Sorted Moran's distribution with elbow")
plt.tight_layout()
plt.show()

# NMF Spatial Factors

In [ ]:
lr_nmf3 = li.multi.nmf(lrdata_nn, n_components=3, inplace=False, random_state=0, max_iter=200, verbose=True)
lr_nmf4 = li.multi.nmf(lrdata_nn, n_components=4, inplace=False, random_state=0, max_iter=200, verbose=True)
lr_nmf5 = li.multi.nmf(lrdata_nn, n_components=5, inplace=False, random_state=0, max_iter=200, verbose=True)
lr_nmf6 = li.multi.nmf(lrdata_nn, n_components=6, inplace=False, random_state=0, max_iter=200, verbose=True)
lr_nmf7 = li.multi.nmf(lrdata_nn, n_components=7, inplace=False, random_state=0, max_iter=200, verbose=True)
lr_nmf8 = li.multi.nmf(lrdata_nn, n_components=8, inplace=False, random_state=0, max_iter=200, verbose=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import cophenet, linkage

# Each result is a tuple: (W, H, errors, rank)
nmf_results = {
    3: lr_nmf3,
    4: lr_nmf4,
    5: lr_nmf5,
    6: lr_nmf6,
    7: lr_nmf7,
    8: lr_nmf8,
}

# Get the raw interaction matrix
X = lrdata_nn.X
if hasattr(X, 'toarray'):
    X = X.toarray()
X = np.clip(X, 0, None).astype(np.float64)

ranks = sorted(nmf_results.keys())
reconstruction_errors = []
sparsity_W_list = []
sparsity_H_list = []
silhouette_scores_list = []
cophenetic_corrs = []

def hoyer_sparsity(x):
    n = x.size
    l1 = np.abs(x).sum()
    l2 = np.sqrt((x**2).sum())
    if l2 == 0:
        return 0.0
    return (np.sqrt(n) - l1 / l2) / (np.sqrt(n) - 1)

for rank in ranks:
    W, H, errors, k = nmf_results[rank]
    
    # Check shapes
    # W: (18374, rank), H: (3014, rank) based on your output
    # Reconstruction: X ≈ W @ H.T
    print(f"Rank {rank}: W={W.shape}, H={H.shape}")
    
    # 1. Reconstruction error
    recon = np.linalg.norm(X - W @ H.T, 'fro')
    reconstruction_errors.append(recon)
    
    # 2. Sparsity
    sparsity_W_list.append(np.mean([hoyer_sparsity(W[:, i]) for i in range(rank)]))
    sparsity_H_list.append(np.mean([hoyer_sparsity(H[:, i]) for i in range(rank)]))
    
    # 3. Silhouette on hard assignments
    labels = np.argmax(W, axis=1)
    if len(np.unique(labels)) > 1:
        sil = silhouette_score(W, labels, metric='cosine')
    else:
        sil = np.nan
    silhouette_scores_list.append(sil)
    
    # 4. Cophenetic correlation
    n_runs = 10
    n_cells = X.shape[0]
    # Subsample to avoid 18k x 18k memory issues
    np.random.seed(42)
    n_sub = min(5000, n_cells)
    idx = np.random.choice(n_cells, n_sub, replace=False)
    X_sub = X[idx, :]
    
    consensus = np.zeros((n_sub, n_sub), dtype=np.float32)
    for run in range(n_runs):
        model = NMF(n_components=rank, init='random', random_state=run, max_iter=500)
        W_run = model.fit_transform(X_sub)
        labels_run = np.argmax(W_run, axis=1)
        conn = (labels_run[:, None] == labels_run[None, :]).astype(np.float32)
        consensus += conn
    consensus /= n_runs
    
    dist = 1 - consensus
    np.fill_diagonal(dist, 0)
    condensed = dist[np.triu_indices(n_sub, k=1)]
    linkage_matrix = linkage(condensed, method='average')
    coph_corr, coph_dist = cophenet(linkage_matrix, condensed)
    cophenetic_corrs.append(coph_corr)
    
    print(f"  recon={recon:.2f}, coph={coph_corr:.3f}, "
          f"sil={sil:.3f}, spW={sparsity_W_list[-1]:.3f}, spH={sparsity_H_list[-1]:.3f}")

# ---- Plot ----
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax in axes.flat:
    ax.set_xticks(ranks)

axes[0, 0].plot(ranks, reconstruction_errors, 'o-', color='#2c7bb6')
axes[0, 0].axvline(x=6, color='red', ls='--', alpha=0.7, label='k=6')
axes[0, 0].set_title('Reconstruction Error')
axes[0, 0].set_xlabel('Rank'); axes[0, 0].legend()

axes[0, 1].plot(ranks, cophenetic_corrs, 's-', color='#d7191c')
axes[0, 1].axvline(x=6, color='red', ls='--', alpha=0.7, label='k=6')
axes[0, 1].set_title('Cophenetic Correlation')
axes[0, 1].set_xlabel('Rank'); axes[0, 1].legend()

axes[1, 0].plot(ranks, silhouette_scores_list, '^-', color='#1a9641')
axes[1, 0].axvline(x=6, color='red', ls='--', alpha=0.7, label='k=6')
axes[1, 0].set_title('Silhouette Score (cosine)')
axes[1, 0].set_xlabel('Rank'); axes[1, 0].legend()

axes[1, 1].plot(ranks, sparsity_W_list, 'D-', color='#fdae61', label='W (cells)')
axes[1, 1].plot(ranks, sparsity_H_list, 'D-', color='#abd9e9', label='H (LRs)')
axes[1, 1].axvline(x=6, color='red', ls='--', alpha=0.7, label='k=6')
axes[1, 1].set_title('Hoyer Sparsity')
axes[1, 1].set_xlabel('Rank'); axes[1, 1].legend()

plt.suptitle('NMF Rank Selection Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('nmf_rank_selection.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for rank in ranks:
    W, H, _, k = nmf_results[rank]
    corr = np.corrcoef(W.T)  # factor-factor correlation
    # Zero out diagonal
    np.fill_diagonal(corr, 0)
    max_corr = np.max(np.abs(corr))
    mean_corr = np.mean(np.abs(corr[np.triu_indices(rank, k=1)]))
    print(f"Rank {rank}: max_factor_corr={max_corr:.3f}, mean_factor_corr={mean_corr:.3f}")

In [ ]:
deltas = np.diff(reconstruction_errors)
print("Marginal improvement per rank increase:")
for i, r in enumerate(ranks[1:]):
    print(f"  {ranks[i]} -> {r}: {deltas[i]:.4f}")

In [ ]:
# Compare factor similarity between k=6 and k=7
W6, H6, _, _ = nmf_results[6]
W7, H7, _, _ = nmf_results[7]

n_top = 50  # top LRs per factor

# Get top LR indices per factor for k=7
top_lrs_7 = [set(np.argsort(H7[:, i])[-n_top:]) for i in range(7)]

# Get top LR indices per factor for k=6
top_lrs_6 = [set(np.argsort(H6[:, i])[-n_top:]) for i in range(6)]

# For each k=7 factor, find its best Jaccard overlap with any k=6 factor
print("k=7 factors matched to k=6:")
for j in range(7):
    overlaps = [len(top_lrs_7[j] & top_lrs_6[i]) / len(top_lrs_7[j] | top_lrs_6[i]) 
                for i in range(6)]
    best_match = np.argmax(overlaps)
    print(f"  Factor {j} -> k6 Factor {best_match} (Jaccard={overlaps[best_match]:.3f})")

# Check if any two k=7 factors map to the same k=6 factor (= splitting)

In [ ]:
W8, H8, _, _ = nmf_results[8]
top_lrs_8 = [set(np.argsort(H8[:, i])[-n_top:]) for i in range(8)]

print("k=8 factors matched to k=6:")
matched_to = []
for j in range(8):
    overlaps = [len(top_lrs_8[j] & top_lrs_6[i]) / len(top_lrs_8[j] | top_lrs_6[i]) 
                for i in range(6)]
    best_match = np.argmax(overlaps)
    matched_to.append(best_match)
    print(f"  Factor {j} -> k6 Factor {best_match} (Jaccard={overlaps[best_match]:.3f})")

# Count how many k=8 factors map to each k=6 factor
from collections import Counter
print("\nk=6 factor split counts:", Counter(matched_to))

In [ ]:
W5, H5, _, _ = nmf_results[5]
top_lrs_5 = [set(np.argsort(H5[:, i])[-n_top:]) for i in range(5)]

print("k=6 factors matched to k=5:")
matched_to_5 = []
for j in range(6):
    overlaps = [len(top_lrs_6[j] & top_lrs_5[i]) / len(top_lrs_6[j] | top_lrs_5[i]) 
                for i in range(5)]
    best_match = np.argmax(overlaps)
    matched_to_5.append(best_match)
    print(f"  k6 Factor {j} -> k5 Factor {best_match} (Jaccard={overlaps[best_match]:.3f})")

from collections import Counter
print("\nk5 factor merge counts:", Counter(matched_to_5))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_top = 50

def compute_jaccard_matrix(top_lrs_a, top_lrs_b):
    """Jaccard overlap between two sets of factor top-LRs"""
    mat = np.zeros((len(top_lrs_a), len(top_lrs_b)))
    for i in range(len(top_lrs_a)):
        for j in range(len(top_lrs_b)):
            inter = len(top_lrs_a[i] & top_lrs_b[j])
            union = len(top_lrs_a[i] | top_lrs_b[j])
            mat[i, j] = inter / union if union > 0 else 0
    return mat

# Top LRs for each rank
all_top_lrs = {}
for rank in [5, 6, 7, 8]:
    W, H, _, _ = nmf_results[rank]
    all_top_lrs[rank] = [set(np.argsort(H[:, i])[-n_top:]) for i in range(rank)]

# Jaccard matrices: k=6 vs others
jac_5v6 = compute_jaccard_matrix(all_top_lrs[5], all_top_lrs[6])
jac_6v7 = compute_jaccard_matrix(all_top_lrs[6], all_top_lrs[7])
jac_6v8 = compute_jaccard_matrix(all_top_lrs[6], all_top_lrs[8])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))


for ax, mat, title, ylabel, xlabel in [
    (axes[0], jac_5v6, 'k=5 vs k=6', 'k=5 factors', 'k=6 factors'),
    (axes[1], jac_6v7, 'k=6 vs k=7', 'k=6 factors', 'k=7 factors'),
    (axes[2], jac_6v8, 'k=6 vs k=8', 'k=6 factors', 'k=8 factors'),
]:
    im = ax.imshow(mat, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    ax.set_yticks(range(mat.shape[0]))
    ax.set_xticks(range(mat.shape[1]))
    
    # Annotate cells
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            color = 'white' if mat[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', 
                    fontsize=8, color=color)

# Replace the colorbar and layout lines with:
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Jaccard Index')
plt.savefig('nmf_factor_stability.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
li.multi.nmf(lrdata_nn, n_components=6, inplace=True, random_state=0, max_iter=200, verbose=True)

In [ ]:
# Extract the variable loadings
lr_loadings = li.ut.get_variable_loadings(lrdata_nn, varm_key='NMF_H').set_index('index')

In [ ]:
factor_scores = li.ut.get_factor_scores(lrdata_nn, obsm_key='NMF_W')

In [ ]:
nmf = sc.AnnData(X=lrdata_nn.obsm['NMF_W'],
                 obs=lrdata_nn.obs,
                 var=pd.DataFrame(index=lr_loadings.columns),
                 uns=lrdata_nn.uns,
                 obsm=lrdata_nn.obsm)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

fig, axes = plt.subplots(1, 2, figsize=(10,5))

# ---------- LEFT PLOT (NO COLORBAR) ----------
sc.pl.embedding(
    nmf,
    basis="spatial_fov",
    color="Factor3",
    size=2,
    color_map="viridis",
    colorbar_loc=None,     # <- remove colorbar here
    legend_loc=None,       # remove discrete legend
    frameon=False,
    ax=axes[0],
    show=False
)

# ---------- RIGHT PLOT (WITH COLORBAR) ----------
sc.pl.embedding(
    nmf,
    basis="spatial_fov",
    color="Factor5",
    size=2,
    color_map="viridis",
    legend_loc=None,
    frameon=False,
    ax=axes[1],
    show=False
)
# Scanpy automatically adds the colorbar for this panel


# ---------- CLEAN UP BORDERS ----------
for ax in axes:
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
nmf[nmf.obs['strict_treg_pt'].isin(['Treg_Early', 'Treg_Late'])]

In [ ]:
cell_type_treg_cci = []
for i,j,k,l in zip(lrdata_nn.obs['cell_type_treg'].tolist(), lrdata_nn.obs['cell_type_general'].tolist(),
                 lrdata_nn.obs['cell_category'].tolist(),lrdata_nn.obs['all_treg_pt'].tolist(), ):
    if k == 'Myeloid':
        if 'DC' in i:
            cell_type_treg_cci.append ('DC')
        elif 'Mo-Mac' in i:
            cell_type_treg_cci.append ('Mo-Mac')
        elif i == ['Mac S+SG+','Mac S+XS-']:
            cell_type_treg_cci.append ('Mac S+SG+')
        else:
            cell_type_treg_cci.append (i)
    elif j == 'Treg':
        cell_type_treg_cci.append (l)
    elif k == 'Connective':
        cell_type_treg_cci.append(j)
    else:
        if k != 'T':
            cell_type_treg_cci.append(k)
        else:
            cell_type_treg_cci.append('Other T')

In [ ]:
lrdata_nn.obs['cell_type_treg_cci']=cell_type_treg_cci

In [ ]:
lrdata_nn.obs['cell_type_treg_cci'].value_counts()

In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

def analyze_nmf_factors(nmf_result, original_adata, n_components, 
                        celltype_column='cell_type_treg_cci',
                        celltypes_of_interest=['Treg_Early', 'Treg_Late', 'CD9+ Mac','Mo-Mac','Mac S+M+P+','DC','Myofibroblast','OGN+RSPO3+ Fib','B','Epithelial','Plasma','NK'],
                        score_threshold=0.5,
                        save_pdf=None,
                        color_vmax_percentile=99,
                        cmap='viridis'):
    """
    Analyze NMF factors: create spatial plots and calculate cell type proportions.
    Ensures all factor plots use the same color range for comparability.

    Parameters:
    -----------
    nmf_result : tuple
        Result from li.multi.nmf() - tuple of (W, H, ...) arrays
    original_adata : AnnData
        Original data with spatial coordinates and cell type annotations
    n_components : int
        Number of NMF components
    celltype_column : str
        Column name in obs containing cell type labels
    celltypes_of_interest : list
        Cell types to calculate proportions for
    score_threshold : float
        Threshold for defining "high score" cells (relative to max score per cell)
    save_pdf : str, optional
        Path to save PDF output (e.g., 'nmf_analysis.pdf')
    color_vmax_percentile : int or None
        If int, use this percentile of W as global vmax (robust to outliers).
        If None, use W.max().
    cmap : str
        Matplotlib colormap name to use for factor plotting.
    """
    
    # Extract W and H from tuple (ignore last 2 items)
    W, H = nmf_result[0], nmf_result[1]
    
    # Create temporary adata with NMF result
    lrdata_tmp = original_adata.copy()
    lrdata_tmp.obsm['NMF_W'] = W
    lrdata_tmp.varm['NMF_H'] = H
    
    # Extract variable loadings and factor scores
    lr_loadings = li.ut.get_variable_loadings(lrdata_tmp, varm_key='NMF_H').set_index('index')
    factor_scores = li.ut.get_factor_scores(lrdata_tmp, obsm_key='NMF_W')
    
    # Create NMF AnnData object
    nmf = sc.AnnData(
        X=W,
        obs=original_adata.obs,
        var=pd.DataFrame(index=lr_loadings.columns),
        uns=original_adata.uns,
        obsm=original_adata.obsm
    )
    
    # Determine global color range across all factors
    # use vmin = 0 (NMF is non-negative); use robust vmax by percentile to avoid extreme outliers
    global_vmin = float(np.nanmin(W)) if np.isfinite(np.nanmin(W)) else 0.0
    if color_vmax_percentile is None:
        global_vmax = float(np.nanmax(W))
    else:
        global_vmax = float(np.nanpercentile(W, color_vmax_percentile))
        # guard: if percentile gives 0 (rare), fall back to max
        if global_vmax <= global_vmin:
            global_vmax = float(np.nanmax(W))
    # If max is equal to min (degenerate), expand slightly to avoid division by zero in plotting
    if np.isclose(global_vmax, global_vmin):
        global_vmax = global_vmin + 1e-6
    
    print(f"\n{'='*60}")
    print(f"NMF Analysis with {n_components} components")
    print(f"Using global color range: vmin={global_vmin:.6g}, vmax={global_vmax:.6g}, cmap={cmap}")
    print(f"{'='*60}\n")
    
    # Open PDF if save_pdf is specified
    if save_pdf:
        pdf = PdfPages(save_pdf)
    
    # Plot factors in pairs — ensure same vmin/vmax/cmap across all plots
    factors = [f'Factor{i+1}' for i in range(n_components)]
    for i in range(0, n_components, 2):
        if i + 1 < n_components:
            # Plot pair of factors
            sc.pl.embedding(
                nmf,
                basis="spatial",
                color=[factors[i], factors[i+1]],
                size=1.4,
                ncols=2,
                show=False if save_pdf else None,
                vmin=global_vmin,
                vmax=global_vmax,
                cmap=cmap
            )
            if save_pdf:
                pdf.savefig(bbox_inches='tight')
                plt.close()
        else:
            # Plot single factor if odd number
            sc.pl.embedding(
                nmf,
                basis="spatial",
                color=[factors[i]],
                size=1.4,
                ncols=1,
                show=False if save_pdf else None,
                vmin=global_vmin,
                vmax=global_vmax,
                cmap=cmap
            )
            if save_pdf:
                pdf.savefig(bbox_inches='tight')
                plt.close()
    
    # Calculate total counts for each cell type (denominator)
    celltype_totals = {}
    for celltype in celltypes_of_interest:
        celltype_totals[celltype] = (original_adata.obs[celltype_column] == celltype).sum()
    
    print("\nTotal cells per cell type:")
    for celltype, total in celltype_totals.items():
        print(f"  {celltype}: {total} cells")
    
    # Analyze cell type composition for each factor
    print("\nCell Type Composition per Factor:")
    print("-" * 60)
    
    factor_compositions_of_celltype = {}  # % of each cell type in factor
    factor_compositions_of_factor = {}    # % of factor that is each cell type
    
    for factor_idx in range(n_components):
        factor_name = f'Factor{factor_idx + 1}'
        
        # Get factor scores for all cells
        scores = W[:, factor_idx]
        
        # Define high-scoring cells (above threshold * max score for each cell across all factors)
        max_scores_per_cell = W.max(axis=1)
        high_score_mask = scores >= (score_threshold * max_scores_per_cell)
        #high_score_mask = scores >= np.percentile(scores, 75)
        
        n_high_score = high_score_mask.sum()
        
        print(f"\n{factor_name}:")
        print(f"  Total high-scoring cells: {n_high_score}")
        
        if n_high_score == 0:
            print(f"  No high-scoring cells found")
            continue
        
        # Get cell type labels for high-scoring cells
        high_score_celltypes = original_adata.obs.loc[high_score_mask, celltype_column]
        
        compositions_of_celltype = {}
        compositions_of_factor = {}
        
        for celltype in celltypes_of_interest:
            # Count how many of this cell type are high-scoring in this factor
            count_in_factor = (high_score_celltypes == celltype).sum()
            
            # Table 1: % of each cell type that is in this factor
            total_of_celltype = celltype_totals[celltype]
            if total_of_celltype > 0:
                pct_of_celltype = count_in_factor / total_of_celltype * 100
            else:
                pct_of_celltype = 0
            compositions_of_celltype[celltype] = pct_of_celltype
            
            # Table 2: % of factor that is this cell type
            pct_of_factor = count_in_factor / n_high_score * 100
            compositions_of_factor[celltype] = pct_of_factor
            
            print(f"  {celltype}:")
            print(f"    {count_in_factor}/{total_of_celltype} of all {celltype} cells ({pct_of_celltype:.2f}%)")
            print(f"    {count_in_factor}/{n_high_score} of factor cells ({pct_of_factor:.2f}%)")
        
        factor_compositions_of_celltype[factor_name] = compositions_of_celltype
        factor_compositions_of_factor[factor_name] = compositions_of_factor
    
    # Create summary dataframes
    composition_df_of_celltype = pd.DataFrame(factor_compositions_of_celltype).T
    composition_df_of_factor = pd.DataFrame(factor_compositions_of_factor).T
    
    print(f"\n{'='*60}")
    print("Summary Table 1: % of each cell type in each factor")
    print("(Row = Factor, Column = Cell Type)")
    print("(Value = % of all cells of that type that are high-scoring in that factor)")
    print(composition_df_of_celltype.round(2))
    
    print(f"\n{'='*60}")
    print("Summary Table 2: Composition of each factor")
    print("(Row = Factor, Column = Cell Type)")
    print("(Value = % of high-scoring cells in that factor that are of that type)")
    print(composition_df_of_factor.round(2))
    
    # Add both summary tables to PDF
    if save_pdf:
        # Table 1: % of cell type
        fig, ax = plt.subplots(figsize=(12, max(6, n_components * 0.6)))
        ax.axis('tight')
        ax.axis('off')
        
        title_text = (f'NMF Analysis with {n_components} components\n'
                     f'% of Each Cell Type in Each Factor\n'
                     f'(% of all cells of that type that are high-scoring in that factor)')
        ax.text(0.5, 0.95, title_text,
                ha='center', va='top', fontsize=12,
                transform=ax.transAxes)
        
        table_data = composition_df_of_celltype.round(2).reset_index()
        table_data.columns = ['Factor'] + list(composition_df_of_celltype.columns)
        
        table = ax.table(cellText=table_data.values,
                        colLabels=table_data.columns,
                        cellLoc='center',
                        loc='center',
                        bbox=[0.05, 0.05, 0.9, 0.75])
        
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2.5)
        
        for i in range(len(table_data.columns)):
            table[(0, i)].set_facecolor('#4472C4')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        for i in range(1, len(table_data) + 1):
            for j in range(len(table_data.columns)):
                if i % 2 == 0:
                    table[(i, j)].set_facecolor('#E7E6E6')
        
        footer_text = "Total cells: " + ", ".join([f"{ct}: {celltype_totals[ct]}" 
                                                     for ct in celltypes_of_interest])
        ax.text(0.5, 0.02, footer_text,
                ha='center', va='bottom', fontsize=9, style='italic',
                transform=ax.transAxes)
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        
        # Table 2: % of factor
        fig, ax = plt.subplots(figsize=(12, max(6, n_components * 0.6)))
        ax.axis('tight')
        ax.axis('off')
        
        title_text = (f'NMF Analysis with {n_components} components\n'
                     f'Composition of Each Factor\n'
                     f'(% of high-scoring cells in that factor that are of that type)')
        ax.text(0.5, 0.95, title_text,
                ha='center', va='top', fontsize=12,
                transform=ax.transAxes)
        
        table_data = composition_df_of_factor.round(2).reset_index()
        table_data.columns = ['Factor'] + list(composition_df_of_factor.columns)
        
        table = ax.table(cellText=table_data.values,
                        colLabels=table_data.columns,
                        cellLoc='center',
                        loc='center',
                        bbox=[0.05, 0.05, 0.9, 0.75])
        
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2.5)
        
        for i in range(len(table_data.columns)):
            table[(0, i)].set_facecolor('#4472C4')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        for i in range(1, len(table_data) + 1):
            for j in range(len(table_data.columns)):
                if i % 2 == 0:
                    table[(i, j)].set_facecolor('#E7E6E6')
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        
        pdf.close()
        print(f"\nPDF saved to: {save_pdf}")
    
    return nmf, composition_df_of_celltype, composition_df_of_factor, lr_loadings


In [ ]:
# Usage - now returns 4 items instead of 3
nmf_results = {
    6: lr_nmf6,
}

all_compositions_of_celltype = {}
all_compositions_of_factor = {}

for n_comp, nmf_res in nmf_results.items():
    nmf_adata, comp_df_celltype, comp_df_factor, loadings = analyze_nmf_factors(
        nmf_res, 
        lrdata_nn, 
        n_components=n_comp,
        save_pdf=f'/data/ydn4687/ibd/cd/figures/nmf_factors_liana_cd/CD_B4/nmf_{n_comp}_components_analysis_new_ct.pdf' #_75th_perc_factor
    )
    all_compositions_of_celltype[n_comp] = comp_df_celltype
    all_compositions_of_factor[n_comp] = comp_df_factor

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages

def plot_lr_correlation_directional(lr_loadings, factor1='Factor3', factor2='Factor5', 
                                    save_pdf_clean=None, save_pdf_labeled=None):

    # ---- Define curated LR pairs with functional categories ----
    lr_categories = {
        'Pro-Inflammatory': [
            'TNFSF13^TNFRSF1A',
            'TNFSF13^TNFRSF14',
            'IL6^IL6ST',
            'IL1A^IL1RAP',
            'CCL2^CCR3',
            'CCL19^CCR7',
            'CCL25^CCR9',
            'CCL25^ACKR4',
            'CCL11^ACKR4',
        ],
        'Immunomodulatory': [
            'LGALS1^PTPRC',
            'IL12A^CD28',
            'CD58^CD2',
            'PTPRC^CD22',
            'CD22^PTPRC'
        ],
        'Pro-Fibrotic/ECM': [
            'TIMP2^MMP2',
            'COL3A1^DDR2',
            'COL17A1^ITGB1',
            'COL9A1^ITGA2',
            'COL27A1^ITGB1',
            'TGM2^SDC4'
        ],
        'Ag Presentation/Treg': [
            'HLA-DRA^CD81',
            'HLA-DRB1^CD81',
            #'HLA-DMB^CD74',
            'HLA-DRA^CD63',
            'ICAM1^MSN'
        ]
    }
    # ---- Collect pairs present in input ----
    all_pairs = []
    pair_categories = {}
    for category, pairs in lr_categories.items():
        for pair in pairs:
            if pair in lr_loadings.index:
                all_pairs.append(pair)
                pair_categories[pair] = category

    # ---- Extract data ----
    plot_data = lr_loadings.loc[all_pairs, [factor1, factor2]].copy()
    plot_data = plot_data.reset_index()
    plot_data.columns = ['LR_pair', factor1, factor2]
    plot_data['Category'] = plot_data['LR_pair'].map(pair_categories)
    plot_data['difference'] = plot_data[factor2] - plot_data[factor1]

    print(f"\nPlotting {len(plot_data)} LR pairs")

    # ---- Color mapping ----
    categories_list = [cat for cat in lr_categories if any(plot_data['Category'] == cat)]
    category_colors = plt.cm.Set2(np.linspace(0, 1, len(categories_list)))
    color_map = dict(zip(categories_list, category_colors))

    # ---- Clean Plot ----
    fig1, ax1 = plt.subplots(figsize=(12, 9))

    for category in categories_list:
        category_data = plot_data[plot_data['Category'] == category]

        # If |difference| < 0.05 → lighter (alpha 0.3)
        alpha_vals = category_data['difference'].abs().apply(lambda d: 0.3 if d < 0.05 else 0.8)

        ax1.scatter(
            category_data[factor1], category_data[factor2],
            s=400,
            alpha=alpha_vals,
            label=category,
            color=color_map[category],
            edgecolors='gray',
            linewidth=2.5
        )

    min_val = min(plot_data[[factor1, factor2]].min())
    max_val = max(plot_data[[factor1, factor2]].max())
    ax1.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.4, linewidth=2.5)

    ax1.set_xlabel(f'{factor1} Loading', fontsize=26)
    ax1.set_ylabel(f'{factor2} Loading', fontsize=26)
    ax1.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
               fontsize=17, framealpha=0.95, edgecolor='black')
    ax1.tick_params(width=2.5, length=8, labelsize=22)

    plt.tight_layout()
    if save_pdf_clean:
        plt.savefig(save_pdf_clean, bbox_inches='tight', dpi=300)
    plt.close(fig1)

    # ---- Labeled Plot ----
    fig2, ax2 = plt.subplots(figsize=(15, 14))

    for category in categories_list:
        category_data = plot_data[plot_data['Category'] == category]

        alpha_vals = category_data['difference'].abs().apply(lambda d: 0.3 if d < 0.05 else 0.8)

        ax2.scatter(
            category_data[factor1], category_data[factor2],
            s=450,
            alpha=alpha_vals,
            label=category,
            color=color_map[category],
            edgecolors='black',
            linewidth=2.5
        )

    ax2.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.4, linewidth=2.5)

    # Number labels
    for idx, row in plot_data.iterrows():
        ax2.text(
            row[factor1], row[factor2], str(idx + 1),
            fontsize=12, ha='center', va='center',
            bbox=dict(boxstyle='circle,pad=0.3', facecolor='white',
                      edgecolor='black', linewidth=1.5)
        )

    ax2.set_xlabel(f'{factor1} Loading', fontsize=26)
    ax2.set_ylabel(f'{factor2} Loading', fontsize=26)
    ax2.set_title('LR Pair Transition with Functional Categories', fontsize=28)
    ax2.legend(loc='upper right', fontsize=17, framealpha=0.95)
    ax2.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()

    # Save labeled version + table
    if save_pdf_labeled:
        pdf = PdfPages(save_pdf_labeled)
        pdf.savefig(fig2, bbox_inches='tight', dpi=300)
        plt.close(fig2)

        # Table
        fig3, ax3 = plt.subplots(figsize=(11, 14))
        ax3.axis('off')
        table_data = plot_data[['LR_pair', 'Category', factor1, factor2, 'difference']].copy()
        table_data.insert(0, 'No.', range(1, len(table_data) + 1))
        table_data = table_data.round(3)

        table = ax3.table(
            cellText=table_data.values,
            colLabels=table_data.columns,
            loc='center',
            cellLoc='left',
            bbox=[0.05, 0.1, 0.9, 0.85]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(11)
        table.scale(1, 2.5)

        pdf.savefig(fig3, bbox_inches='tight', dpi=300)
        plt.close(fig3)
        pdf.close()

    return plot_data


In [ ]:
# Usage:
comparison_data = plot_lr_correlation_directional(
    lr_loadings, 
    factor1='Factor3', 
    factor2='Factor5',
    save_pdf_clean='/data/ydn4687/ibd/cd/figures/CD_B4_morans_factor3_vs_factor5_clean.pdf',
    save_pdf_labeled='/data/ydn4687/ibd/cd/figures/CD_B4_morans_factor3_vs_factor5_ref.pdf'
)


In [ ]:
imm_lrs_sc = pd.read_csv('/path/to/scrna/cd/nichenet_destab_late_treg_ligand_strength_imm_logpr_best_sender.csv')

In [ ]:
imm_lrs_sc=imm_lrs_sc.sort_values('priority_rank_geom',ascending=False)

In [ ]:
from kneed import KneeLocator
import numpy as np

x = range(0,imm_lrs_sc.shape[0])  # your values
y = imm_lrs_sc['priority_rank_geom'].tolist()

# ignore the first N very-steep points
N = 10   # try 10, 15, 20 depending on how much early drop you want to ignore

x_sub = x[N:]
y_sub = y[N:]

kl = KneeLocator(
    x_sub,
    y_sub,
    curve='convex',        # your shape is still convex + decreasing overall
    direction='decreasing'
)

print("Raw elbow in sub-curve:", kl.elbow)

# map back to original index if you want position in [1..len(y)]
elbow_global = kl.elbow
print("Global elbow (same as x index):", elbow_global)


In [ ]:
sel_lrs = imm_lrs_sc[:25]
sel_lrs['from_to'] = [i+'^'+j for i,j in zip(sel_lrs['from'].tolist(), sel_lrs['to'].tolist())]
sel_lrs